# Copia de trabajo — Sección 3

Esta es **tu** copia: aquí escribes los ejercicios y guardas tus figuras. La versión canónica vive en la raíz del repositorio
(`seccion3_revision_vcf.ipynb`, commit `e433007`) y no se edita: si la modificas, el próximo `git pull` choca.

Se puede abrir desde `tareas/` o desde la raíz: la primera celda de código ajusta la carpeta sola.

# Sección 3 · Un VCF real: anatomía, calidad y decisiones

*Cómo leer un archivo de secuenciación real, encontrar sus problemas y justificar cada filtro*

## Objetivos de aprendizaje

Al terminar podrás:

1. **Leer** un VCF real y convertirlo en una matriz de dosis (0, 1, 2, `NA`) alineada con sus metadatos.
2. **Resumir** el archivo en una ficha técnica: muestras, SNP, datos faltantes y SNP sin variación.
3. **Reconocer tres señales de alerta** en datos reales: exceso de heterocigotos (parálogos), pérdida de heterocigotos por baja profundidad y réplicas.
4. **Medir lo que cuesta** cada filtro y **calcular diversidad** sobre el archivo ya limpio.
5. **Mirar la estructura** de los sitios con un PCA.
6. **Justificar** por escrito una decisión de filtrado para cada alerta.

## Los datos

`datos_ejemplo/lenga_real_1782snp.vcf.gz` es un subconjunto del VCF publicado por Sekely et al. (2024) sobre *Nothofagus pumilio*: **207 muestras** (200 árboles, 10 por sitio en 20 sitios, **6 réplicas** y **1 muestra** casi sin datos) y **1 782 SNP**, uno por contig. Cada genotipo trae `GT:AD:DP:GQ`.

**Qué no repetimos.** La importación y los datos faltantes (Sección 1) y el cálculo de Ho, He y Fis (Sección 2): aquí usamos la función `diversity_by_pop()` que ya conoces.

> **Antes de empezar, anota dos predicciones:** ¿qué porcentaje de genotipos crees que falta? ¿Cuántos de los 1 782 SNP crees que **no varían** entre estos 207 árboles?

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4.2, repr.plot.res = 110)
suppressPackageStartupMessages({ library(vcfR); library(ggplot2) })
if (!file.exists("datos_ejemplo") && file.exists("../datos_ejemplo")) setwd("..")   # si abriste el notebook desde una subcarpeta
source("R/helpers.R")                                   # funciones de la Sección 2 (diversity_by_pop)

VCF  <- "datos_ejemplo/lenga_real_1782snp.vcf.gz"
META <- "datos_ejemplo/lenga_real_muestras.csv"
stopifnot(file.exists(VCF), file.exists(META))          # si falla, abre Jupyter desde la carpeta del repositorio

---
# Parte demostrativa

## 1. Un VCF es un archivo de texto

- Las líneas que empiezan con `##` son el **encabezado**; la que empieza con `#CHROM` nombra las columnas.
- Cada línea siguiente es **un SNP**; desde la décima columna hay **un árbol por columna**.
- En este archivo cada genotipo tiene la forma `GT:AD:DP:GQ`: genotipo, lecturas por alelo, profundidad y calidad (escala phred).

*Predice:* ¿cuántas líneas de datos esperas?

In [ ]:
lineas <- readLines(gzfile(VCF))                        # el archivo está comprimido (.gz)
c(lineas = length(lineas),
  encabezado = sum(startsWith(lineas, "##")),     # + la línea #CHROM que nombra columnas
  snps = sum(!startsWith(lineas, "#")))
cat(substr(lineas[startsWith(lineas, "#CHROM")], 1, 150), "\n\n")      # nombres de las columnas
cat(substr(lineas[!startsWith(lineas, "#")][1], 1, 250), "\n")          # el primer SNP

**Léelo con calma.** En la primera línea de datos identifica el contig y la posición, el alelo de referencia (`REF`) y el alternativo (`ALT`), y los primeros genotipos con sus cuatro campos.

## 2. Leer con `vcfR` y construir la matriz de dosis

Extraemos tres matrices (genotipo, profundidad y calidad) y convertimos cada genotipo en el **número de copias del alelo alternativo**: `0/0` → 0, `0/1` → 1, `1/1` → 2, `./.` → `NA`. Los genotipos con `|` (por ejemplo `0|1`) son los mismos con fase conocida; para contar da igual.

In [ ]:
vcf <- read.vcfR(VCF, verbose = FALSE)
gt  <- extract.gt(vcf, "GT")                            # genotipos, SNP x árbol
dp  <- t(extract.gt(vcf, "DP", as.numeric = TRUE))      # profundidad, árbol x SNP
gq  <- t(extract.gt(vcf, "GQ", as.numeric = TRUE))      # calidad,     árbol x SNP
table(gt, useNA = "ifany")                              # ¿qué genotipos hay?

dosis <- c("0/0" = 0, "0/1" = 1, "1/0" = 1, "1/1" = 2)
# Red de seguridad: si apareciera un genotipo raro (1/2, .|1), quedaría en NA en silencio.
raros <- sum(!is.na(gt) & !(sub("|", "/", gt, fixed = TRUE) %in% names(dosis)))
if (raros > 0) warning("Hay ", raros, " genotipos fuera de 0/0, 0/1, 1/1 y ./. : revisar antes de seguir.")
G <- t(matrix(dosis[sub("|", "/", gt, fixed = TRUE)], nrow(gt), dimnames = dimnames(gt)))
colnames(G) <- colnames(dp) <- colnames(gq) <- paste(getCHROM(vcf), getPOS(vcf), sep = "_")
stopifnot(identical(dim(dp), dim(G)))
G[1:5, 1:6]                                             # filas = árboles, columnas = SNP

In [ ]:
meta <- read.csv(META, stringsAsFactors = FALSE)
meta <- meta[match(rownames(G), meta$id), ]
stopifnot(all(meta$id == rownames(G)))                  # ¡el orden de los datos y las etiquetas debe calzar!

table(meta$sitio_n, useNA = "ifany")                    # árboles por sitio (NA = muestras sin ficha)
rownames(G)[is.na(meta$sitio_n)]                        # ¿cuáles son?

**Regla de oro.** Si `meta` estuviera en otro orden, todo se calcularía sin error y estaría mal. Por eso el `stopifnot()`.

Aparecen **7 muestras sin ficha** en el material publicado. No es un error: es lo que pasa con datos reales. En la sección 6 descubrirás qué son seis de ellas.

## 3. La ficha técnica

Llamamos **p** a la frecuencia del alelo alternativo (`p = media de la dosis / 2`) y **MAF** a la del alelo menor, `min(p, 1 − p)`. Un SNP con MAF = 0 no varía entre estos árboles.

In [ ]:
falt_ind <- rowMeans(is.na(G))                         # fracción de faltantes por árbol
p   <- colMeans(G, na.rm = TRUE) / 2
maf <- pmin(p, 1 - p)
polimorfico <- !is.na(maf) & maf > 0                    # ¿el SNP varía entre los árboles?

c(muestras = nrow(G), snp = ncol(G), sitios = length(unique(na.omit(meta$sitio_n))),
  sin_ficha = sum(is.na(meta$sitio_n)),
  faltantes_pct = round(100 * mean(is.na(G)), 1),
  snp_sin_variacion = sum(!polimorfico),
  snp_maf_menor_5pct = sum(polimorfico & maf < 0.05))

head(sort(falt_ind, decreasing = TRUE), 8)              # los árboles con más faltantes

**Discute.** ¿Coincide con tus predicciones? ¿Qué muestra destaca por sus faltantes y qué harías con ella? Recuerda que **la MAF depende del tamaño de muestra**: con solo 207 árboles aparecen como «sin variación» SNP que sí varían en el archivo completo (en la corrida de referencia, 497 de los 1 782 aquí, frente a solo 53 con las 502 muestras). Filtrar por MAF con pocos individuos elimina variantes reales.

## 4. Primera señal de alerta: exceso de heterocigotos

Un SNP con **casi todos** los árboles heterocigotos suele ser un **parálogo** (dos copias del genoma colapsadas en un contig): inflan la heterocigosidad y hunden la Fis. El método HDplot del estudio descarta los SNP con una proporción de heterocigotos **H mayor que 0,6**. Pero con pocos árboles H sube por azar: veamos cuánto.

In [ ]:
het_snp <- colMeans(G == 1, na.rm = TRUE)              # H: proporción de heterocigotos por SNP
hist(het_snp, breaks = 40, col = "darkorange", main = "", xlab = "H por SNP"); abline(v = 0.6, lty = 2, col = "red")
sospechosos <- names(which(het_snp > 0.6)); length(sospechosos)

# ¿cuánto puede subir H solo por azar en un SNP normal (50 % de heterocigotos esperados)?
n <- c(10, 25, 50, 100, 207)
data.frame(arboles = n, prob_azar_H_mayor_0.6 = round(1 - pbinom(floor(0.6 * n), n, 0.5), 3))

In [ ]:
# ¿Se repite el exceso en los sitios? H de cada sospechoso, sitio por sitio
sitios <- sort(unique(na.omit(meta$sitio_n)))
H_sitio <- sapply(sitios, function(k) colMeans(G[which(meta$sitio_n == k), sospechosos, drop = FALSE] == 1, na.rm = TRUE))
table(sitios_con_H_alto = rowSums(H_sitio > 0.6, na.rm = TRUE))

**Discute.** Con muchos árboles, superar H = 0,6 por azar casi no ocurre; con 10 sí. Un parálogo se comporta mal **en todas las poblaciones**: los sospechosos que superan 0,6 en casi todos los sitios son los candidatos sólidos. ¿Los eliminarías a todos, solo a esos o a ninguno?

## 5. Segunda señal de alerta: profundidad y calidad

Con **pocas lecturas** el programa puede leer solo una de las dos copias de un heterocigoto y llamarlo homocigoto (*allelic dropout*). Predicción comprobable: la proporción de heterocigotos debería ser **menor cuando la profundidad es baja**.

In [ ]:
llamado <- !is.na(G)                                    # solo donde hubo genotipo
d <- data.frame(dp = dp[llamado], het = G[llamado] == 1)
d$rango <- cut(d$dp, c(-1, 3, 7, 11, 19, Inf), labels = c("0-3", "4-7", "8-11", "12-19", "20+"))
tab <- tapply(d$het, d$rango, mean)
round(tab, 3)
barplot(tab, col = "steelblue", xlab = "profundidad (DP) del genotipo", ylab = "proporción de heterocigotos")

**Discute.** ¿Sube la proporción de heterocigotos con la profundidad? Si no se filtrara, Ho bajaría y Fis subiría sin que haya endogamia. Por eso el estudio exige **DP ≥ 8 y GQ ≥ 20** (un error de 1 % o menos). Midamos lo que cuesta.

In [ ]:
conserva <- llamado & dp >= 8 & gq >= 20               # TRUE = el genotipo se mantiene
cat("Genotipos llamados que pasan a faltante:",
    round(100 * (1 - sum(conserva, na.rm = TRUE) / sum(llamado)), 1), "%\n")

falt_final <- 100 * (1 - rowSums(conserva, na.rm = TRUE) / ncol(G))
summary(falt_final)
barplot(sort(falt_final), col = "grey60", border = NA, ylab = "% faltante tras el filtro", xlab = "árboles (ordenados)")

**Idea clave.** El filtro **no reparte la pérdida de forma pareja**: hay árboles que quedan casi vacíos. Eso arrastra una cadena: más faltantes por árbol → se eliminan muestras → cambia el *n* por sitio → cambian He y Fis. Un filtro no se elige por «lo que se ve bien», sino **midiendo qué se pierde**.

## 6. Tercera señal de alerta: réplicas

Algunos árboles están genotipados dos veces (réplicas técnicas). No son individuos distintos: si se quedan, duplican un árbol y sesgan las frecuencias. Y sirven para algo: la **discrepancia** entre las dos copias **mide el error de genotipado**. Se detectan comparando genotipos: la **identidad** entre dos muestras es la proporción de SNP en los que tienen el mismo genotipo.

In [ ]:
ok <- !is.na(G)
iguales <- Reduce(`+`, lapply(0:2, function(v) tcrossprod(((G == v) & ok) * 1)))   # SNP con el mismo genotipo, para cada par
comparables <- tcrossprod(ok * 1)                                                  # SNP con dato en ambas muestras
ident <- iguales / comparables
diag(ident) <- NA; ident[comparables < 500] <- NA       # pares con muy pocos SNP comparables no cuentan

pares <- which(ident > 0.95, arr.ind = TRUE)
pares <- pares[pares[, 1] < pares[, 2], , drop = FALSE]  # cada par una sola vez
data.frame(muestra_1 = rownames(G)[pares[, 1]], muestra_2 = rownames(G)[pares[, 2]],
           identidad = round(ident[pares], 4), discrepancia_pct = round(100 * (1 - ident[pares]), 2))

**Idea clave.** Las seis parejas son las muestras sin ficha: en su nombre llevan una `R`. El nombre lo insinúa; la identidad genotípica lo **demuestra**. La discrepancia (unos pocos %) estima el error de genotipado. Cuidado: entre árboles **distintos** la identidad también es alta (mediana ≈ 0,91 y máximo ≈ 0,94 en este panel), porque la mayoría de los SNP varían poco. Por eso el umbral 0,95 solo funciona porque las réplicas quedan claramente por encima: el margen es de centésimas, no de décimas, y depende del panel.

**Decisión:** se conserva la copia que tiene ficha (la original) y se informa la discrepancia en los Métodos.

## 7. El archivo limpio y la diversidad por sitio

Con las tres decisiones tomadas (DP y GQ, réplicas y muestra vacía) recién tiene sentido calcular diversidad.

In [ ]:
quitar <- which(is.na(meta$sitio_n) | falt_ind > 0.9)   # las réplicas (sin ficha) y la muestra casi vacía
table(muestras_quitadas_por_par = rowSums(matrix(c(pares) %in% quitar, ncol = 2)))   # cada par pierde exactamente una
keep <- setdiff(seq_len(nrow(G)), quitar)

G_limpio    <- G[keep, ]
G_limpio[!conserva[keep, ]] <- NA                       # aplica DP >= 8 y GQ >= 20
meta_limpio <- meta[keep, ]
# ¿El SNP todavía varía? Se mira en las FRECUENCIAS, no en los valores de dosis:
# un SNP en el que todos los árboles son heterocigotos tiene una sola dosis (1)
# pero es máximamente polimórfico (p = 0,5, MAF = 0,5). Con unique() se perderían 21 de esos.
p_limpio   <- colMeans(G_limpio, na.rm = TRUE) / 2
maf_limpio <- pmin(p_limpio, 1 - p_limpio)
pol        <- !is.na(maf_limpio) & maf_limpio > 0

c(muestras = nrow(G_limpio), snp_variables = sum(pol), faltantes_pct = round(100 * mean(is.na(G_limpio)), 1))

In [ ]:
div <- diversity_by_pop(G_limpio[, pol], meta_limpio$sitio_n)            # la función de la Sección 2
div$lat <- tapply(meta_limpio$lat, meta_limpio$sitio_n, mean)[div$pop]
round(colMeans(div[, c("Ho", "He", "Fis")]), 3)
round(range(div$Fis), 2)

**Discute.** En la corrida de referencia los 20 sitios salen con **Fis negativo**: más heterocigotos de los que predice Hardy-Weinberg. ¿Es biología (una especie estrictamente alógama) o son los parálogos que todavía quedan adentro? Probemos quitar SNP según su H.

In [ ]:
het_limpio <- colMeans(G_limpio == 1, na.rm = TRUE)
round(t(sapply(c(1, 0.6, 0.5, 0.4), function(corte) {
  sel <- pol & (is.na(het_limpio) | het_limpio <= corte)
  d <- diversity_by_pop(G_limpio[, sel], meta_limpio$sitio_n)
  c(corte_H = corte, snp = sum(sel), Ho = mean(d$Ho), He = mean(d$He), Fis = mean(d$Fis))
})), 3)

**Idea clave.** Al quitar los SNP con H > 0,6 (el umbral del estudio) el exceso baja mucho: **la mayor parte venía de posibles parálogos**. Con cortes más estrictos Fis se acerca a cero, pero ese camino es **circular**: elegir los SNP por su heterocigosidad observada y luego calcular Fis con esa misma heterocigosidad la empuja hacia cero por construcción, y además descarta SNP legítimos con He alta. Los autores reportan Ho > He en todos los sitios incluso después de eliminar parálogos, así que un resto pequeño de exceso es compatible con la biología. Ni Ho ni He son «el» valor de la diversidad: cambian con el conjunto de SNP y los filtros, y siempre se reportan junto a ellos.

Y un detalle que vale para tu propio código: **hay SNP en los que todos los árboles son heterocigotos** (21 en este archivo limpio, con H = 1,0). Son una sola dosis y, a la vez, lo más polimórfico que existe. Si los buscaras con `unique()` los perderías **en silencio**, y con ellos la señal más fuerte de parálogos.

## 8. ¿Se separan los sitios? PCA de los árboles

El PCA resume cientos de SNP en dos ejes de máxima variación; cada árbol es un punto. **No es un test**: es una fotografía para mirar los datos. Dos detalles prácticos: no admite datos faltantes (los rellenamos con la media del SNP) y los primeros ejes suelen explicar **poco** porcentaje.

In [ ]:
Gi <- apply(G_limpio[, pol], 2, function(x) { x[is.na(x)] <- mean(x, na.rm = TRUE); x })   # faltantes -> media del SNP
pc <- prcomp(Gi)
ve <- round(100 * pc$sdev^2 / sum(pc$sdev^2), 1)

d_pc <- data.frame(PC1 = pc$x[, 1], PC2 = pc$x[, 2], lat = meta_limpio$lat)
ggplot(d_pc, aes(PC1, PC2, colour = lat)) + geom_point(size = 2.5) + scale_colour_viridis_c() +
  labs(x = paste0("PC1 (", ve[1], " %)"), y = paste0("PC2 (", ve[2], " %)"), colour = "latitud")
# La correlación que importa es la del PC1 *medio de cada sitio* con la latitud:
# a nivel de árbol el ruido individual la diluye.
pc1_sitio  <- tapply(pc$x[, 1], meta_limpio$sitio_n, mean)
lat_sitio  <- tapply(meta_limpio$lat, meta_limpio$sitio_n, mean)
c(varianza_PC1_pct = ve[1],
  cor_arbol_latitud = round(cor(pc$x[, 1], meta_limpio$lat), 2),
  cor_sitio_latitud = round(cor(pc1_sitio, lat_sitio), 2))

**Discute.** El PC1 explica un porcentaje pequeño de la variación y aun así ordena los árboles según la latitud (en la corrida de referencia, con una correlación fuerte). Tres ideas:

- **Poca varianza explicada no significa «no hay estructura»**: importa el patrón de los individuos en el plano.
- Un eje **no es «la latitud»**: es la dirección de mayor variación, que puede alinearse con ella.
- Es una pista, no una demostración: ¿qué estadístico pedirías para afirmar que los sitios están diferenciados?

## Resumen de la parte demostrativa

1. Se lee el VCF, se arma la matriz de dosis y **siempre** se comprueba el orden contra los metadatos.
2. La ficha técnica resume faltantes y SNP sin variación; la MAF depende del tamaño de muestra.
3. **Tres alertas:** H alto (parálogos, pero también azar con pocos árboles), poca profundidad (pérdida de heterocigotos) y réplicas (su discrepancia mide el error).
4. Cada filtro cuesta datos: se mide lo que se pierde y se reporta.
5. Sobre el archivo limpio, la diversidad cambia con el conjunto de SNP; filtrar por H hasta que Fis valga cero es circular.
6. El PCA es una fotografía, no una prueba.

---
# Parte de ejercicios

Resuelve en las celdas siguientes y anota tus **decisiones** (umbrales y por qué): serán tu sección de *Métodos*.

### E1. Completa la ficha técnica
Agrega tres campos: **SNP con más de 20 % de faltantes**, **árboles con más de 50 % de faltantes** y **SNP con H mayor que 0,6**. ¿Alguno te preocupa?

In [ ]:
# TU CÓDIGO AQUÍ
# Pista: sum(colMeans(is.na(G)) > 0.2), sum(falt_ind > 0.5), sum(het_snp > 0.6, na.rm = TRUE)

### E2. El costo de los umbrales de profundidad y calidad
Calcula el **porcentaje de genotipos llamados que se pierden** con DP ≥ 4, 8 y 12 combinados con GQ ≥ 20 y 30. ¿Qué combinación elegirías y por qué?

In [ ]:
# TU CÓDIGO AQUÍ
# Pista: expand.grid(dp_min = c(4, 8, 12), gq_min = c(20, 30)) y, para cada fila,
#        100 * (1 - sum(llamado & dp >= dp_min & gq >= gq_min, na.rm = TRUE) / sum(llamado))

### E3. ¿Son parálogos de verdad?
Con `H_sitio` de la sección 4: ¿cuántos sospechosos tienen H > 0,6 en **al menos 15 de los 20 sitios**? ¿Los eliminarías solo a ellos? Justifica con lo que viste sobre el azar.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista: sum(rowSums(H_sitio > 0.6, na.rm = TRUE) >= 15)

### E4. Error de genotipado
Con las parejas de la sección 6, calcula la **discrepancia media** entre réplicas y su original. ¿Qué umbral de identidad separa limpiamente réplicas de árboles distintos? (Mira la distribución de `ident`.)

In [ ]:
# TU CÓDIGO AQUÍ
# Pista: mean(100 * (1 - ident[pares]))   y   hist(ident[upper.tri(ident)], breaks = 50)

### E5. PCA con SNP comunes
Repite el PCA usando solo los SNP con **MAF ≥ 0,05** (calculada sobre `G_limpio`). ¿Cambia el porcentaje de varianza del PC1? ¿Cambia su correlación con la latitud? ¿Qué te dice sobre depender del conjunto de SNP?

In [ ]:
# TU CÓDIGO AQUÍ
# Pista: p2 <- colMeans(G_limpio, na.rm = TRUE) / 2;  comunes <- pol & pmin(p2, 1 - p2) >= 0.05
#        y repite el bloque del PCA con G_limpio[, comunes]

### E6. Tres alertas y una decisión
Con lo que encontraste, completa la tabla. Una alerta por fila; cada decisión debe incluir **un umbral y su justificación**.

| # | Alerta (qué viste y dónde) | Posible causa | Decisión (con umbral) | Justificación |
|---|---|---|---|---|
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |

---
## Errores y dudas frecuentes

- **«No encuentra `datos_ejemplo/...` o `R/helpers.R`»** → abriste Jupyter desde otra carpeta; ábrelo desde la carpeta del repositorio.
- **«`readLines` lee basura»** → el archivo está comprimido: usa `readLines(gzfile(VCF))`.
- **«`stopifnot` falla al unir los metadatos»** → los nombres de las muestras no calzan; compara `rownames(G)` con `meta$id`.
- **«`extract.gt()` devuelve SNP × árboles»** → sí: por eso se transpone con `t()` para que `G`, `dp` y `gq` sean árboles × SNP.
- **«La Fis sale muy negativa»** → suele ser exceso de heterocigotos por parálogos: revisa las secciones 4 y 7, sin bajar el corte de H tanto que Fis valga cero por construcción.

## Tarea (se revisa al comienzo de la próxima sección)

1. Entrega tu **ficha técnica** completa (E1) y la tabla de **tres alertas** (E6).
2. Escribe en **5 líneas** los umbrales de filtrado que usarás y su justificación (faltantes, DP, GQ, H y qué haces con las réplicas): será el borrador de tus Métodos.
3. Interpreta el **PCA** en tres líneas: qué muestra, qué no permite concluir y qué estadístico pedirías para afirmar que los sitios están diferenciados.

**Para tu proyecto con microsatélites:** las revisiones de faltantes, réplicas y Ho, He, Fis se repiten; en lugar de parálogos el problema típico son los alelos nulos, y no existen profundidad ni calidad por genotipo.

**Para profundizar:** [Introducción al VCF (Browning)](https://faculty.washington.edu/browning/beagle/intro-to-vcf.html) · [Visualizar un VCF con vcfR](https://knausb.github.io/vcfR_documentation/visualization_2.html) · [Datos de lenga en Zenodo](https://zenodo.org/records/10018374) · [Preprint de Sekely et al.](https://www.biorxiv.org/content/10.1101/2023.04.28.538677v1)